In [ ]:
import numpy as np
import glob
import torch
import json
import torch.nn.functional as F
import nibabel as nib
import huggingface_hub

from totalsegmentator.python_api import totalsegmentator
from scipy.ndimage import label, binary_closing, binary_erosion, center_of_mass
from matplotlib import pyplot as plt
from PIL import Image

from modeling.BaseModel import BaseModel
from modeling import build_model
from utilities.distributed import init_distributed
from utilities.arguments import load_opt_from_config_files
from utilities.constants import BIOMED_CLASSES


from inference_utils.inference import interactive_infer_image, interactive_infer_image_all
from inference_utils.output_processing import dice_volume, iou_volume, hausdorff_distance_volume
from inference_utils.processing_utils import read_nifti_only, process_intensity_image, resize_image, volume_trimmer

C:\Users\Toni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Deformable Transformer Encoder is not available.


C:\Users\Toni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\kornia\feature\lightglue.py:30: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
with open('tokens.json') as f:
    tokens = json.load(f)

In [3]:
HF_TOKEN = tokens['hugging_face']

huggingface_hub.login(HF_TOKEN)

### Model Setup

In [4]:
opt = load_opt_from_config_files(["configs/biomedparse_inference.yaml"])
opt = init_distributed(opt)

# Load model from pretrained weights
# pretrained_pth = 'pretrained\\biomedparse_v3_abdomen_tumor.pt'
# pretrained_pth = 'model_state_dict.pt'
pretrained_pth = 'hf_hub:microsoft/BiomedParse'

model = BaseModel(opt, build_model(opt)).from_pretrained(pretrained_pth).eval().cuda()
with torch.no_grad():
    model.model.sem_seg_head.predictor.lang_encoder.get_text_embeddings(BIOMED_CLASSES + ["background"], is_eval=True)
print(pretrained_pth)


C:\Users\Toni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
$UNUSED$ criterion.empty_weight, Ckpt Shape: torch.Size([17])


hf_hub:microsoft/BiomedParse


### Utility Functions

In [ ]:
!pip install pydicom nibabel SimpleITK


def inference_nifti(image, text_prompts, is_CT, site=None):
    test_3D = np.zeros((image.shape[2], image.shape[0], image.shape[1]))

    for slice_iter in range(image.shape[2]):
        
        # Resize and padding
        image_array = process_intensity_image(image[: ,: , slice_iter], is_CT, site)
            
        # Get prediction
        pred_mask = interactive_infer_image(model, Image.fromarray(image_array), text_prompts)
        # pred_mask = (pred_mask > 0.5).astype(np.uint8)

        # Resize to original size
        resize_image = resize_image(pred_mask, image.shape[0], image.shape[1])

        # Stack back together
        test_3D[slice_iter] = resize_image.squeeze()


    # print(f"Patient {file_path[14:20]} is complete")

    assert test_3D.shape[1] == test_3D.shape[2]

    final_img = test_3D.transpose(1,2,0)

    return final_img



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Toni\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
labels = {'tumor': 1, 'aorta': 2, 'right kidney': 3, 'left kidney': 4, 'duodenum': 5, 'pancreas': 6, 'liver':7, 'spleen':8, 'stomach':9, 'gallbladder':10, 'left adrenal gland':11, 'right adrenal gland':12, 'esophagus':13, 'postcava': 14}

In [ ]:
!pip install pydicom nibabel SimpleITK


def inference_nifti_all(image, is_CT, image_type, p_value_threshold, site=None):
    test_3D = np.zeros((image.shape[2], image.shape[0], image.shape[1]))

    for slice_iter in range(image.shape[2]):
        # Resize and padding
        image_array = process_intensity_image(image[: ,: , slice_iter], is_CT, site)

        predictions = interactive_infer_image_all(model, Image.fromarray(image_array), image_type, p_value_threshold)

        targets = list(predictions.keys())
        pred_mask = [predictions[t] for t in targets]

        mask_combined = np.zeros((pred_mask[0].shape[0], pred_mask[0].shape[1]))

        for i, mask in enumerate(pred_mask):
            print(mask.shape)
            if targets[i] != 'tumor':
                mask[mask == 1] = labels[targets[i]]
                mask_combined = mask_combined + mask
        
        # Resize to original size
        resize_image = resize_image(mask_combined, image.shape[0], image.shape[1])

        # Stack back together
        test_3D[slice_iter] = resize_image.squeeze()

    # print(f"Patient {file_path[14:20]} is complete")

    assert test_3D.shape[1] == test_3D.shape[2]

    final_img = test_3D.transpose(1,2,0)
    
    return final_img



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Toni\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
PDAC_patients = [100002, 100005, 100011, 100030, 100033, 100043, 100050, 100060, 100074, 100082, 100091, 100096, 100101, 100102, 100124, 100127, 100134]
len(PDAC_patients)

17

In [ ]:
for p in PDAC_patients:
    img_paths = f'data//CT//img//{p}*'
    is_CT = True

    results = []
    for img_path in glob.iglob(img_paths):
        print(img_path)
        print("Locating kidneys ...")

        input = nib.load(img_path)
        image = input.get_fdata()
        output = totalsegmentator(input, roi_subset_robust=['kidney_right', 'kidney_left'])

        output = output.get_fdata()

        first, last = volume_trimmer(output)

        print('\n')
        print("Pancreatic tumor segmentation ...")
        print('\n')

        image_window = np.zeros((image.shape[0], image.shape[1], image.shape[2]))

        pancreas_pred = inference_nifti_all(image[:,:, first:last], True, "CT-Abdomen", None, site='abdomen')
        pancreas_pred = (pancreas_pred > 0.5).astype(np.uint8)
        image_window[:, :, first:last] = pancreas_pred

        image_window = image_window + output
        image_window[image_window > 1] = 0

        final_img = nib.Nifti1Image(image_window, input.affine)
        nib.save(final_img, f'organ_inference/CT_patient_{img_path[14:20]}_all_BM.nii.gz')

   

data//CT//img\100002_00001_0000.nii.gz
Locating kidneys ...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 8.32s
Predicting...


100%|██████████| 6/6 [00:40<00:00,  6.71s/it]


  Predicted in 76.20s
Resampling...
  cropping from (512, 512, 561) to (278, 154, 142)
Resampling...
  Resampled in 0.55s
Predicting part 1 of 1 ...


100%|██████████| 2/2 [00:03<00:00,  1.64s/it]


  Predicted in 18.54s
Resampling...


Pancreatic tumor segmentation ...


(512, 512, 98)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(

e:\University\Thesis\Sandbox\BiomedParse\inference_utils\output_processing.py:45: RuntimeWarning: Mean of empty slice.
  return [mask[mask>=128].mean()/256, img[:,:,0][mask>=128].mean()/256,
C:\Users\Toni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\Toni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
e:\University\Thesis\Sandbox\BiomedParse\inference_utils\output_processing.py:46: RuntimeWarning: Mean of empty slice.
  img[:,:,1][mask>=128].mean()/256, img[:,:,2][mask>=128].mean()/256]


(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)


In [ ]:
for p in PDAC_patients:
    img_paths = f'data//CT//img//{p}*'
    is_CT = True

    results = []
    for img_path in glob.iglob(img_paths):
        print(img_path)
        print("Locating kidneys ...")

        input = nib.load(img_path)
        image = input.get_fdata()
        output = totalsegmentator(input, roi_subset_robust=['kidney_right', 'kidney_left'])


        output = output.get_fdata()

        first, last = volume_trimmer(output)

        print('\n')
        print("Pancreatic tumor segmentation ...")
        print('\n')

        image_window = np.zeros((image.shape[0], image.shape[1], image.shape[2]))

        pancreas_pred = inference_nifti(image[:,:, first:last], ["tumor"], is_CT=is_CT, site="abdomen")
        pancreas_pred = (pancreas_pred > 0.5).astype(np.uint8)
        image_window[:, :, first:last] = pancreas_pred

        # Inverse Kidney mask over BioMeds predicitons to remove some hallucinations
        image_window = image_window + output
        image_window[image_window > 1] = 0

        final_img = nib.Nifti1Image(image_window, input.affine)
        nib.save(final_img, f'organ_inference/CT_patient_{img_path[14:20]}_all_BM.nii.gz')

   

100%|██████████| 4/4 [00:00<00:00, 71.43it/s]


In [ ]:
for patient in PDAC_patients:

    path_label = f'data//CT//gt//{patient}_00001.nii.gz'
    label, nii = read_nifti_only(path_label)

    # One hot encode multiple classes
    unique_labels = np.unique(label)

    buffer = int(label.shape[2] * 0.05)
    first, last = volume_trimmer(label)

    label_one_hot = F.one_hot(torch.tensor(label).long(), num_classes=-1)

    label_one_hot = label_one_hot[:, :, first:last, 1]

    path_pred = f'results_pub//CT_patient_TS_v3_{patient}.nii.gz'
    pred, nii = read_nifti_only(path_pred)

    pred = pred[:, :, first:last]
    pred = torch.tensor(pred)


    # 1 is currently PDAC lessions, watch out for which label we are calculating
    if 1 not in unique_labels:
         continue
    else:
        dice = dice_volume(torch.permute(label_one_hot, (2, 0, 1)), torch.permute(pred, (2, 0, 1)))
        iou = iou_volume(torch.permute(label_one_hot, (2, 0, 1)), torch.permute(pred, (2, 0, 1)))
        hausdorff = hausdorff_distance_volume(torch.permute(label_one_hot, (2, 0, 1)), torch.permute(pred, (2, 0, 1)))

        with open("metrics_v3.txt", "a") as f:
            f.write(f"3D_DICE score for patient {patient} is : {dice}\n")
            f.write(f"3D_IoU score for patient {patient} is : {iou}\n")
            f.write(f"3D_HD score for patient {patient} is : {hausdorff}\n")
            f.write("\n")

    print(f"Patient {patient} done!")

Patient 100002 done!
Patient 100005 done!
Patient 100011 done!
